In [1]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from collections import Counter
import pandas as pd
import numpy as np
import os
from IPython.display import HTML, display
import textwrap

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("itachi9604/disease-symptom-description-dataset")

print("Path to dataset files:", path)

c:\Users\koley\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\koley\.cache\kagglehub\datasets\itachi9604\disease-symptom-description-dataset\versions\2


In [3]:
print(os.listdir(path))

['dataset.csv', 'Symptom-severity.csv', 'symptom_Description.csv', 'symptom_precaution.csv']


In [4]:
dfD = pd.read_csv(os.path.join(path, 'symptom_Description.csv'))
dfP = pd.read_csv(os.path.join(path, 'symptom_precaution.csv'))
dfS = pd.read_csv(os.path.join(path, 'Symptom-severity.csv'))

In [5]:
dfD.head()

,Disease,Description
0,Drug Reaction,An adverse drug reaction (ADR) is an injury ca...
1,Malaria,An infectious disease caused by protozoan para...
2,Allergy,An allergy is an immune system response to a f...
3,Hypothyroidism,"Hypothyroidism, also called underactive thyroi..."
4,Psoriasis,Psoriasis is a common skin disorder that forms...


In [6]:
dfP.head()

,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Drug Reaction,stop irritation,consult nearest hospital,stop taking drug,follow up
1,Malaria,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,Allergy,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,Hypothyroidism,reduce stress,exercise,eat healthy,get proper sleep
4,Psoriasis,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths


In [7]:
dfS.tail()

,Symptom,weight
128,inflammatory_nails,2
129,blister,4
130,red_sore_around_nose,2
131,yellow_crust_ooze,3
132,prognosis,5


In [8]:
df = pd.read_csv("Training.csv")
dft = pd.read_csv("Testing.csv")

In [9]:
df.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis,Unnamed: 133
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Fungal infection,NaN


In [10]:
print(df[df["prognosis"] == "hepatitis A"].head(1))

     itching  skin_rash  nodal_skin_eruptions  continuous_sneezing  shivering  \
190        0          0                     0                    0          0   

     chills  joint_pain  stomach_pain  acidity  ulcers_on_tongue  ...  \
190       0           1             0        0                 0  ...   

     scurring  skin_peeling  silver_like_dusting  small_dents_in_nails  \
190         0             0                    0                     0   

     inflammatory_nails  blister  red_sore_around_nose  yellow_crust_ooze  \
190                   0        0                     0                  0   

       prognosis  Unnamed: 133  
190  hepatitis A           NaN  

[1 rows x 134 columns]


In [11]:
df = df.drop("Unnamed: 133", axis = 1)

In [12]:
df["prognosis"].value_counts()

prognosis
Fungal infection                           120
Hepatitis C                                120
Hepatitis E                                120
Alcoholic hepatitis                        120
Tuberculosis                               120
Common Cold                                120
Pneumonia                                  120
Dimorphic hemmorhoids(piles)               120
Heart attack                               120
Varicose veins                             120
Hypothyroidism                             120
Hyperthyroidism                            120
Hypoglycemia                               120
Osteoarthristis                            120
Arthritis                                  120
(vertigo) Paroymsal  Positional Vertigo    120
Acne                                       120
Urinary tract infection                    120
Psoriasis                                  120
Hepatitis D                                120
Hepatitis B                                120
All

In [13]:
Xtrain = df.drop("prognosis", axis = 1)
Ytrain = df["prognosis"]

In [14]:
Xtrain.columns.value_counts()

itching                 1
movement_stiffness      1
muscle_pain             1
irritability            1
depression              1
                       ..
constipation            1
back_pain               1
pain_behind_the_eyes    1
loss_of_appetite        1
yellow_crust_ooze       1
Name: count, Length: 132, dtype: int64

In [15]:
Xtrain

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,pus_filled_pimples,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4915,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4916,0,1,0,0,0,0,0,0,0,0,...,1,1,1,0,0,0,0,0,0,0
4917,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4918,0,1,0,0,0,0,1,0,0,0,...,0,0,0,1,1,1,1,0,0,0


In [16]:
Xtrain.columns = Xtrain.columns.str.replace("_"," ")

In [17]:
Xtrain.head()

,itching,skin rash,nodal skin eruptions,continuous sneezing,shivering,chills,joint pain,stomach pain,acidity,ulcers on tongue,...,pus filled pimples,blackheads,scurring,skin peeling,silver like dusting,small dents in nails,inflammatory nails,blister,red sore around nose,yellow crust ooze
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
print(Ytrain)

0                              Fungal infection
1                              Fungal infection
2                              Fungal infection
3                              Fungal infection
4                              Fungal infection
                         ...                   
4915    (vertigo) Paroymsal  Positional Vertigo
4916                                       Acne
4917                    Urinary tract infection
4918                                  Psoriasis
4919                                   Impetigo
Name: prognosis, Length: 4920, dtype: object


In [19]:
word = df["prognosis"].str.split().sum()
Counter(word).most_common(20)

[('Hepatitis', 480),
 ('infection', 240),
 ('hepatitis', 240),
 ('Fungal', 120),
 ('Allergy', 120),
 ('GERD', 120),
 ('Chronic', 120),
 ('cholestasis', 120),
 ('Drug', 120),
 ('Reaction', 120),
 ('Peptic', 120),
 ('ulcer', 120),
 ('diseae', 120),
 ('AIDS', 120),
 ('Diabetes', 120),
 ('Gastroenteritis', 120),
 ('Bronchial', 120),
 ('Asthma', 120),
 ('Hypertension', 120),
 ('Migraine', 120)]

In [20]:
le = LabelEncoder()
Ytrain = le.fit_transform(Ytrain)
Ytrain = pd.DataFrame(Ytrain)

In [21]:
Ytrain.value_counts()

0     120
21    120
23    120
24    120
25    120
26    120
27    120
28    120
29    120
30    120
31    120
32    120
33    120
34    120
35    120
36    120
37    120
38    120
39    120
22    120
20    120
1     120
19    120
2     120
3     120
4     120
5     120
6     120
7     120
8     120
9     120
10    120
11    120
12    120
13    120
14    120
15    120
16    120
17    120
18    120
40    120
Name: count, dtype: int64

In [22]:
dft.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,0,0,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Allergy
2,0,0,0,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,GERD
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Chronic cholestasis
4,1,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,Drug Reaction


In [23]:
df

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4915,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,(vertigo) Paroymsal Positional Vertigo
4916,0,1,0,0,0,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,Acne
4917,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Urinary tract infection
4918,0,1,0,0,0,0,1,0,0,0,...,0,0,1,1,1,1,0,0,0,Psoriasis


In [24]:
Xtest = dft.drop(["prognosis"], axis = 1)
Ytest = dft["prognosis"]

Ytest = le.transform(Ytest)
Ytest = pd.DataFrame(Ytest)

Ytest

,0
0,15
1,4
2,16
3,9
4,14
5,33
6,1
7,12
8,17
9,6


In [25]:
Xtest.columns = Xtest.columns.str.replace("_"," ")

In [26]:
print(Xtrain.shape)
print(Ytrain.shape)
print(Xtest.shape)
print(Ytest.shape)

(4920, 132)
(4920, 1)
(42, 132)
(42, 1)


In [27]:
model = RandomForestClassifier(
    n_estimators = 100,
    max_depth = 5,
    min_samples_leaf = 2,
    random_state = 42
)

model.fit(Xtrain, Ytrain)

c:\Users\koley\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1351: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier(max_depth=5, min_samples_leaf=2, random_state=42)

In [28]:
Ypred = model.predict(Xtest)
print(accuracy_score(Ytest, Ypred))

0.9761904761904762


In [29]:
Yogpred = le.inverse_transform(Ypred)
print(Yogpred)

['Fungal infection' 'Allergy' 'GERD' 'Chronic cholestasis' 'Drug Reaction'
 'Peptic ulcer diseae' 'AIDS' 'Diabetes ' 'Gastroenteritis'
 'Bronchial Asthma' 'Hypertension ' 'Migraine' 'Cervical spondylosis'
 'Paralysis (brain hemorrhage)' 'Jaundice' 'Malaria' 'Chicken pox'
 'Dengue' 'Typhoid' 'hepatitis A' 'Hepatitis B' 'Hepatitis C'
 'Hepatitis D' 'Hepatitis E' 'Alcoholic hepatitis' 'Tuberculosis'
 'Common Cold' 'Pneumonia' 'Dimorphic hemmorhoids(piles)' 'Heart attack'
 'Varicose veins' 'Hypothyroidism' 'Hyperthyroidism' 'Hypoglycemia'
 'Osteoarthristis' 'Arthritis' '(vertigo) Paroymsal  Positional Vertigo'
 'Acne' 'Urinary tract infection' 'Psoriasis' 'Impetigo' 'Impetigo']


In [30]:
Xtrain = df.drop("prognosis", axis=1)
Ytrain = df["prognosis"]
Xtrain.columns = Xtrain.columns.str.replace("_"," ")

le = LabelEncoder()
Ytrain = le.fit_transform(Ytrain)

Ytrain = pd.DataFrame(Ytrain)

Xtest = dft.drop("prognosis", axis=1)
Ytest = dft["prognosis"]
Xtest.columns = Xtest.columns.str.replace("_"," ")

Ytest = le.transform(Ytest)
Ytest = pd.DataFrame(Ytest)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=2,
    random_state=42
)

model.fit(Xtrain, Ytrain)

Ypred = model.predict(Xtest)

print(accuracy_score(Ytest, Ypred))

Yogpred = le.inverse_transform(Ypred)
print(Yogpred)


c:\Users\koley\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1351: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


0.9761904761904762
['Fungal infection' 'Allergy' 'GERD' 'Chronic cholestasis' 'Drug Reaction'
 'Peptic ulcer diseae' 'AIDS' 'Diabetes ' 'Gastroenteritis'
 'Bronchial Asthma' 'Hypertension ' 'Migraine' 'Cervical spondylosis'
 'Paralysis (brain hemorrhage)' 'Jaundice' 'Malaria' 'Chicken pox'
 'Dengue' 'Typhoid' 'hepatitis A' 'Hepatitis B' 'Hepatitis C'
 'Hepatitis D' 'Hepatitis E' 'Alcoholic hepatitis' 'Tuberculosis'
 'Common Cold' 'Pneumonia' 'Dimorphic hemmorhoids(piles)' 'Heart attack'
 'Varicose veins' 'Hypothyroidism' 'Hyperthyroidism' 'Hypoglycemia'
 'Osteoarthristis' 'Arthritis' '(vertigo) Paroymsal  Positional Vertigo'
 'Acne' 'Urinary tract infection' 'Psoriasis' 'Impetigo' 'Impetigo']


In [31]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(Xtrain.columns.tolist(), "symptoms.pkl")


['symptoms.pkl']

In [32]:
Xtrain.columns

Index(['itching', 'skin rash', 'nodal skin eruptions', 'continuous sneezing',
       'shivering', 'chills', 'joint pain', 'stomach pain', 'acidity',
       'ulcers on tongue',
       ...
       'pus filled pimples', 'blackheads', 'scurring', 'skin peeling',
       'silver like dusting', 'small dents in nails', 'inflammatory nails',
       'blister', 'red sore around nose', 'yellow crust ooze'],
      dtype='object', length=132)

In [33]:
import os
from flask import Flask, request, jsonify
import pandas as pd
import joblib
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

# Load ML artifacts
model = joblib.load("model.pkl")
le = joblib.load("label_encoder.pkl")
symptoms = joblib.load("symptoms.pkl")

# # Load CSV files (same directory)
# dfD = pd.read_csv("symptom_Description.csv")
# dfP = pd.read_csv("symptom_precaution.csv")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.json
    selected_symptoms = data.get("symptoms", [])

    input_data = pd.DataFrame([[0]*len(symptoms)], columns=symptoms)

    for s in selected_symptoms:
        if s in input_data.columns:
            input_data.loc[0, s] = 1

    pred = model.predict(input_data)
    disease = le.inverse_transform(pred)[0]

    description = dfD[dfD["Disease"] == disease]["Description"].iloc[0]

    prec_row = dfP[dfP["Disease"] == disease].iloc[0]
    precautions = [
        prec_row["Precaution_1"],
        prec_row["Precaution_2"],
        prec_row["Precaution_3"],
        prec_row["Precaution_4"]
    ]

    return jsonify({
        "disease": disease,
        "about": description,
        "precautions": precautions
    })

if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Feb/2026 15:19:05] "OPTIONS /predict HTTP/1.1" 200 -
127.0.0.1 - - [12/Feb/2026 15:19:05] "POST /predict HTTP/1.1" 200 -


In [ ]:
for col in Xtrain.columns:
  print(col)

itching
skin rash
nodal skin eruptions
continuous sneezing
shivering
chills
joint pain
stomach pain
acidity
ulcers on tongue
muscle wasting
vomiting
burning micturition
spotting  urination
fatigue
weight gain
anxiety
cold hands and feets
mood swings
weight loss
restlessness
lethargy
patches in throat
irregular sugar level
cough
high fever
sunken eyes
breathlessness
sweating
dehydration
indigestion
headache
yellowish skin
dark urine
nausea
loss of appetite
pain behind the eyes
back pain
constipation
abdominal pain
diarrhoea
mild fever
yellow urine
yellowing of eyes
acute liver failure
fluid overload
swelling of stomach
swelled lymph nodes
malaise
blurred and distorted vision
phlegm
throat irritation
redness of eyes
sinus pressure
runny nose
congestion
chest pain
weakness in limbs
fast heart rate
pain during bowel movements
pain in anal region
bloody stool
irritation in anus
neck pain
dizziness
cramps
bruising
obesity
swollen legs
swollen blood vessels
puffy face and eyes
enlarged thyroi